# Stage-z drift QC

Every acquired movie writes a HAL `.off` focus-lock log alongside its image file (same directory, same stem). It's a whitespace-delimited table, one row per frame, with columns `frame offset power stage-z good-offset`. The focus lock is expected to hold `stage-z` constant for a whole FOV's stack.

This notebook, for every round and every FOV:
- reads the `.off` sidecar's `stage-z` column,
- checks whether every row is the same value,
- if not, records the first row's value plus the min/max (should be rare),
- caches the result to `analysis/stage_z_summary.csv` so re-running this notebook only reads *new* `.off` files, never re-reading ones already summarized,
- plots the first-frame `stage-z` value against a continuous FOV order (cells block, then hyb01, hyb02, ...) so drift over the whole acquisition is visible at a glance.

This is a one-shot notebook — re-run any of its cells at any point during or after acquisition to refresh the plot with whatever has been written so far.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/analysis/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.analysis.stage_z import update_stage_z_cache, assign_x_positions, round_label

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL is writing

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

CACHE_PATH = config.analysis_dir / "stage_z_summary.csv"

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Positions tag: {POSITIONS_TAG}")
print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Cache  : {CACHE_PATH}")

## 3 — Read / update the stage-z cache

Re-run this cell any time — it only reads `.off` files not already in the cache.

In [ ]:
n_before = 0
if CACHE_PATH.exists():
    import pandas as pd
    n_before = len(pd.read_csv(CACHE_PATH))

cache = update_stage_z_cache(config, meta, CACHE_PATH)
print(f"Stage-z cache: {len(cache)} FOV\u00d7series entries total "
      f"({len(cache) - n_before} newly read this run).")

drifted = cache[~cache["all_same"]]
if drifted.empty:
    print("Every FOV's stage-z was constant across its whole frame stack.")
else:
    print(f"{len(drifted)} FOV(s) had a non-constant stage-z within their stack:")
    display(
        drifted[["round_id", "fov_id", "series", "first_stage_z", "min_stage_z", "max_stage_z"]]
        .sort_values(["round_id", "fov_id"])
    )

## 4 — Plot drift over the acquisition

One point per FOV (its first frame's `stage-z`), laid out as one continuous axis: the cells block, then hyb01, hyb02, ... in round order, FOVs in ID order within each round.

In [ ]:
plot_df = assign_x_positions(cache)

fig, ax = plt.subplots(figsize=(14, 4))
ax.scatter(plot_df["x"], plot_df["first_stage_z"], s=6, alpha=0.6)

round_starts = plot_df.groupby("round_id")["x"].min().sort_index()
for x in round_starts.values[1:]:
    ax.axvline(x - 0.5, color="lightgray", linewidth=0.7, zorder=0)
ax.set_xticks(round_starts.values)
ax.set_xticklabels([round_label(meta, rid) for rid in round_starts.index], rotation=90)

ax.set_xlabel("Round (FOVs in order within each round)")
ax.set_ylabel("stage-z (\u00b5m) \u2014 first frame of each FOV")
ax.set_title("Stage-z drift across the acquisition")
fig.tight_layout()
plt.show()